# P02. Parquet, contrato y reejecución

9 de octubre de 2026, sesión 3

Preparado por el PhD Esteban Hernández, CyberColombia.

Estos ejercicios usan tres muestras reales de 1.000 filas. Las primeras filas sirven para probar código, sin representatividad estadística. Los archivos originales permanecen intactos. No se requiere GPU ni una cuenta externa. Instalar requirements.txt y requirements_pqrs.txt y seleccionar el kernel del mismo entorno.


**Entorno:** Ubuntu-26.04 sobre WSL. Clonar siempre `main` en `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. Activar `.venv` desde la raíz y abrir Jupyter desde `kit/`. Seguir [la guía WSL](../../docs/instalacion-wsl.md).

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import sys, json
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "kit", Path.cwd() / "big-data-postgraduate" / "kit"]
KIT = next((p.resolve() for p in candidates if (p / "pqrs_talleres.py").exists()), None)
assert KIT is not None, "Abrir desde bigdata/, la raíz del repositorio, kit/ o kit/Notebooks/"
sys.path.insert(0, str(KIT))
import pqrs_talleres as lab
a = SimpleNamespace(datos=None, completo=False, bloque=500, out=KIT / "salidas" / "notebooks", repeticiones=3)
a.out.mkdir(parents=True, exist_ok=True)


In [ ]:
a.bloque = 500
r1 = lab.formatos(a)
r2 = lab.formatos(a)
assert r1["filas"] == r2["filas"] == 3000
assert r1["firma_contenido"] == r2["firma_contenido"]
assert r2["columnas_producto"] == 16
r2

## Fallo controlado
Retirar una columna en memoria. El contrato debe detener la publicación. Conservar el original completo de 38 columnas.

In [ ]:
f, p = lab.seleccion(a)[0]
chunk = next(lab.leer(f, p, 100))
try:
    lab.normalizar(chunk.drop(columns=[chunk.columns[0]]), f, 0)
except ValueError as e:
    print("Contrato detectó el cambio:", e)
else:
    raise AssertionError("El contrato debía fallar")

In [ ]:
particiones = list((a.out / "pqrs_particionado").rglob("*.parquet"))
assert len(particiones) == 8
[(p.parent.relative_to(a.out).as_posix(), p.stat().st_size) for p in particiones]

## Entrega
Contrato de columnas y unidades, proyección de 16 campos, traza del fallo y comprobación de reejecución. Las carpetas representan año y mes. La sustitución de salidas evita anexar filas en una reejecución; no equivale a una transacción atómica de todo el producto.